# 08 · Developing Data Exchange Pipelines

> Companion notebook to `docs/data-exchange/` from the Learn OpenUSD curriculum.

## Learning objectives
- Define what OpenUSD data exchange is and identify the four common implementation shapes (importers, exporters, standalone converters, file format plugins).
- Apply the two-phase *extract-then-transform* pattern to a realistic OBJ→USD converter.
- Extract polygonal geometry and `UsdPreviewSurface` materials from an OBJ source into an in-memory stage.
- Transform the extracted stage by setting a `defaultPrim` and supporting a user-selectable up-axis export option.
- Validate the resulting asset with `UsdUtils.ComplianceChecker` (the Python surface of `usdchecker`) and, when available, Omniverse `AssetValidator`.

## How to use this notebook
Run cells top-to-bottom. Generated files land in `./_assets/` next to this notebook. Safe to delete that folder any time. No `assimp_py` install is required — we ship a tiny inline OBJ/MTL parser so every cell runs with just `usd-core`.

## Prerequisites
Notebooks 00–07 (in particular *Stage Setting*, *Scene Description Blueprints*, and *Asset Structure*).

In [1]:
import sys
print("Python:", sys.version.split()[0])
from pxr import Usd, Sdf, Gf, UsdGeom, UsdShade
print("pxr import OK.")
from pathlib import Path
NB_DIR = Path.cwd()
(NB_DIR / "_assets").mkdir(exist_ok=True)
print("_assets/ ready at:", NB_DIR / "_assets")

Python: 3.12.9
pxr import OK.
_assets/ ready at: /Users/stranzersweb/Projects/LearnOpenUSD/notebooks/.exec/08_data_exchange/_assets


In [2]:
try:
    from lousd.utils.helperfunctions import create_new_stage
    from lousd.utils.visualization import DisplayUSD, DisplayCode
    print("lousd helpers loaded.")
    _HAVE_LOUSD = True
except ImportError:
    import os
    from pathlib import Path as _P
    def create_new_stage(relative_file_path: str):
        from pxr import Usd, Sdf
        ident = os.path.join(os.getcwd(), relative_file_path)
        found = Sdf.Layer.Find(identifier=ident)
        if found:
            return Usd.Stage.Open(found.identifier)
        return Usd.Stage.CreateNew(relative_file_path)
    def DisplayUSD(path, show_usd_code=True):
        from pxr import Usd
        print(f"--- {path} ---")
        print(Usd.Stage.Open(str(path)).ExportToString(addSourceFileComment=False))
    def DisplayCode(path):
        print(f"--- {path} ---")
        print(_P(path).read_text())
    print("Using fallback helpers.")
    _HAVE_LOUSD = False

lousd helpers loaded.


## Stage the source assets

The docs exercises operate on `shapes.obj` / `shapes.mtl` under `docs/exercise_content/data_exchange/`. We copy those files into `_assets/` so the notebook is self-contained and portable. If the repo layout changes, we degrade gracefully by falling back to a tiny synthetic OBJ built inline — the remainder of the lessons still work.


In [3]:
import shutil
from pathlib import Path

ASSETS = Path.cwd() / "_assets"
ASSETS.mkdir(exist_ok=True)

SRC_OBJ = Path.cwd().parents[0] / "docs" / "exercise_content" / "data_exchange" / "shapes.obj"
SRC_MTL = Path.cwd().parents[0] / "docs" / "exercise_content" / "data_exchange" / "shapes.mtl"
DST_OBJ = ASSETS / "shapes.obj"
DST_MTL = ASSETS / "shapes.mtl"

copied = False
try:
    if SRC_OBJ.exists() and SRC_MTL.exists():
        shutil.copy2(SRC_OBJ, DST_OBJ)
        shutil.copy2(SRC_MTL, DST_MTL)
        copied = True
        print(f"Copied exercise assets:\n  {DST_OBJ}\n  {DST_MTL}")
    else:
        print("Exercise assets not found at expected repo path; will synthesize a minimal OBJ below.")
except Exception as exc:
    print(f"Could not copy exercise assets ({exc}); will synthesize a minimal OBJ below.")

if not DST_OBJ.exists():
    # Minimal single-cube OBJ with a single material so the rest of the notebook still runs.
    DST_OBJ.write_text("""mtllib shapes.mtl
o Cube
v -1 -1 -1
v -1  1 -1
v  1  1 -1
v  1 -1 -1
v -1 -1  1
v -1  1  1
v  1  1  1
v  1 -1  1
usemtl Material_001
f 1 2 3 4
f 5 8 7 6
f 1 5 6 2
f 4 3 7 8
f 2 6 7 3
f 1 4 8 5
""")
    DST_MTL.write_text("""newmtl Material_001
Ns 200.0
Ka 0.0 0.0 0.0
Kd 0.8 0.2 0.2
Ks 0.5 0.5 0.5
Ke 0.0 0.0 0.0
""")
    print("Wrote synthetic shapes.obj / shapes.mtl under _assets/.")

print("shapes.obj size:", DST_OBJ.stat().st_size, "bytes")

Exercise assets not found at expected repo path; will synthesize a minimal OBJ below.
Wrote synthetic shapes.obj / shapes.mtl under _assets/.
shapes.obj size: 193 bytes


## 8.1 What is data exchange?

OpenUSD data exchange answers two questions: **"How do I get my data into USD?"** and **"How do I get my data out of USD?"** There are four common implementation shapes for bolting OpenUSD onto an existing tool or format:

- **Importers** — translate USD into a DCC's runtime format (*File > Import*).
- **Exporters** — translate the DCC's runtime format into USD (*File > Export*).
- **Standalone converters** — a script, executable, or microservice dedicated to translating one file format to another. Directionality can be one-way or two-way; document it clearly.
- **File format plugins** — unique to USD: they let a stage compose a foreign format (or even a database/procedural source) directly as a reference, payload, or sublayer, translating on the fly.

### Challenges
- Data exchange is almost always lossy. Not every model maps cleanly.
- Not every consumer needs every piece of data (a CAD app rarely needs animation; a mobile renderer cannot handle film-quality geometry).
- No single content structure suits every organization.

### Two-phase data exchange
The module proposes a two-phase design loosely inspired by ETL:

1. **Extract** — translate the source to USD *as directly as possible*, preserving source fidelity.
2. **Transform** — one or more optional steps that apply user export options, restructure content to fit a pipeline, or optimize for a downstream runtime.

Keeping these phases separate means third-party developers can layer their own transforms onto your raw extraction without reverse-engineering your output.

### 8.1.1 Anatomy of a converter

The docs build up a standalone OBJ→USD converter called `obj2usd.py`. Rather than running it as a subprocess (which would require `assimp_py`), we bring its structure into the notebook as in-memory functions — `extract()` and `transform()` called from `main()` — and supply our own tiny OBJ/MTL parser so the lesson is self-contained. Below is the boilerplate the docs start with.

In [4]:
import argparse
import logging
import math
from enum import Enum
from pathlib import Path

from pxr import Usd, UsdGeom

logger = logging.getLogger("obj2usd")
logging.basicConfig(level=logging.INFO)

class UpAxis(Enum):
    Y = UsdGeom.Tokens.y
    Z = UsdGeom.Tokens.z

    def __str__(self):
        return self.value

print("UpAxis options:", [str(a) for a in UpAxis])

UpAxis options: ['Y', 'Z']


**What just happened:** we set up the converter scaffolding — logging, a `UpAxis` enum mirroring `UsdGeom.Tokens.y`/`.z`, and the standard imports. In the docs, a `main()` function at the bottom wires up `argparse` and calls `extract()` then `transform()`; we will do the same, but invoke `main()` directly from a cell.

### 8.1.2 A minimal OBJ/MTL parser (stand-in for Assimp)

The docs use `assimp_py` to get a `scene` object with `meshes` and `materials`. We replicate just enough of that shape for the lessons to land: a namedtuple-based `Scene` whose `meshes` have `name`, `vertices`, `indices` (per-face), `normals`, `material_index`, and whose `materials` is a list of dicts with `NAME`, `COLOR_DIFFUSE`, `COLOR_EMISSIVE`, `COLOR_SPECULAR`, `SHININESS`. This keeps downstream code identical to the docs.

In [5]:
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Optional, Tuple

@dataclass
class Mesh:
    name: str
    vertices: List[Tuple[float, float, float]] = field(default_factory=list)
    indices: List[List[int]] = field(default_factory=list)  # per-face index lists
    normals: List[Tuple[float, float, float]] = field(default_factory=list)
    material_index: int = 0

@dataclass
class Scene:
    meshes: List[Mesh] = field(default_factory=list)
    materials: List[Dict] = field(default_factory=list)

def _parse_mtl(path: Path) -> List[Dict]:
    materials: List[Dict] = []
    current: Optional[Dict] = None
    if not path.exists():
        return materials
    for raw in path.read_text().splitlines():
        line = raw.strip()
        if not line or line.startswith("#"):
            continue
        parts = line.split()
        key = parts[0]
        if key == "newmtl":
            current = {
                "NAME": parts[1],
                "COLOR_DIFFUSE": (0.8, 0.8, 0.8),
                "COLOR_EMISSIVE": (0.0, 0.0, 0.0),
                "COLOR_SPECULAR": (0.5, 0.5, 0.5),
                "SHININESS": 50.0,
            }
            materials.append(current)
        elif current is None:
            continue
        elif key == "Kd" and len(parts) >= 4:
            current["COLOR_DIFFUSE"] = tuple(float(x) for x in parts[1:4])
        elif key == "Ke" and len(parts) >= 4:
            current["COLOR_EMISSIVE"] = tuple(float(x) for x in parts[1:4])
        elif key == "Ks" and len(parts) >= 4:
            current["COLOR_SPECULAR"] = tuple(float(x) for x in parts[1:4])
        elif key == "Ns" and len(parts) >= 2:
            current["SHININESS"] = float(parts[1])
    return materials

def parse_obj(path: Path) -> Scene:
    """Very small OBJ reader: handles v / vn / f / o / usemtl / mtllib."""
    path = Path(path)
    scene = Scene()
    # Global vertex pools (OBJ uses 1-based, global indices).
    v_pool: List[Tuple[float, float, float]] = []
    vn_pool: List[Tuple[float, float, float]] = []
    mtl_path: Optional[Path] = None
    current: Optional[Mesh] = None
    mat_name_to_idx: Dict[str, int] = {}
    # A mesh's own local vertex list mapping global->local.
    local_lookup: Dict[int, int] = {}
    per_mesh_normals: List[Tuple[float, float, float]] = []

    def start_mesh(name: str) -> Mesh:
        nonlocal current, local_lookup, per_mesh_normals
        current = Mesh(name=name)
        scene.meshes.append(current)
        local_lookup = {}
        per_mesh_normals = []
        return current

    for raw in path.read_text().splitlines():
        line = raw.strip()
        if not line or line.startswith("#"):
            continue
        parts = line.split()
        key = parts[0]
        if key == "mtllib":
            mtl_path = path.parent / parts[1]
        elif key == "o":
            start_mesh(parts[1] if len(parts) > 1 else f"mesh_{len(scene.meshes)}")
        elif key == "v":
            v_pool.append(tuple(float(x) for x in parts[1:4]))
        elif key == "vn":
            vn_pool.append(tuple(float(x) for x in parts[1:4]))
        elif key == "usemtl":
            if current is None:
                start_mesh("mesh_0")
            # material index will be resolved after mtl is parsed below
            current.material_index = -1
            current._pending_mat = parts[1]  # type: ignore[attr-defined]
        elif key == "f":
            if current is None:
                start_mesh("mesh_0")
            face_indices: List[int] = []
            for tok in parts[1:]:
                # token looks like v, v/vt, v//vn, or v/vt/vn (1-based, global)
                segs = tok.split("/")
                gv = int(segs[0]) - 1
                if gv not in local_lookup:
                    local_lookup[gv] = len(current.vertices)
                    current.vertices.append(v_pool[gv])
                face_indices.append(local_lookup[gv])
                if len(segs) == 3 and segs[2]:
                    gn = int(segs[2]) - 1
                    # one normal per vertex in the mesh (simplification)
                    while len(current.normals) <= local_lookup[gv]:
                        current.normals.append((0.0, 0.0, 0.0))
                    current.normals[local_lookup[gv]] = vn_pool[gn]
            current.indices.append(face_indices)

    # Load materials from mtllib, then resolve each mesh's material_index.
    if mtl_path is not None:
        scene.materials = _parse_mtl(mtl_path)
        mat_name_to_idx = {m["NAME"]: i for i, m in enumerate(scene.materials)}
    for m in scene.meshes:
        pending = getattr(m, "_pending_mat", None)
        if pending is None:
            m.material_index = 0
        else:
            m.material_index = mat_name_to_idx.get(pending, 0)
            delattr(m, "_pending_mat")
    # Backfill a default material if the MTL was empty.
    if not scene.materials:
        scene.materials = [{
            "NAME": "Material_default",
            "COLOR_DIFFUSE": (0.8, 0.8, 0.8),
            "COLOR_EMISSIVE": (0.0, 0.0, 0.0),
            "COLOR_SPECULAR": (0.5, 0.5, 0.5),
            "SHININESS": 50.0,
        }]
    return scene

_scene_preview = parse_obj(DST_OBJ)
print(f"Parsed {len(_scene_preview.meshes)} mesh(es) and {len(_scene_preview.materials)} material(s)")
for m in _scene_preview.meshes:
    print(f"  mesh '{m.name}': {len(m.vertices)} verts, {len(m.indices)} faces, material_index={m.material_index}")

Parsed 1 mesh(es) and 1 material(s)
  mesh 'Cube': 8 verts, 6 faces, material_index=0


**What just happened:** our stand-in parser produced a `Scene` that exposes exactly the fields the docs' `extract()` reads (`scene.meshes`, `mesh.vertices`, `mesh.indices`, `mesh.normals`, `mesh.material_index`, `scene.materials[i]["NAME"]`, etc.). That means the converter code we write next is byte-for-byte the same as the course material — no special casing.

## 8.2 Data extraction

The extract phase is about fidelity: make the USD output look as close to the source format as you reasonably can. Keeping extraction faithful has three payoffs:

- Output is legible, which makes debugging and round-tripping easier.
- It provides a stable foundation every transformation step can build on.
- Third-party developers can plug in their own transforms on top of your raw extraction.

A **conceptual data mapping document** is worth producing alongside the code — it records which concepts in the source format map to which schemas in USD, and where the gaps are. For OBJ→USD, meshes map to `UsdGeom.Mesh`, per-face vertex data maps to flattened `faceVertexIndices` + `faceVertexCounts`, and OBJ `.mtl` materials map to `UsdShade.Material` + `UsdPreviewSurface` shaders.

### 8.2.1 Extracting geometry

For each mesh in the scene we:
1. Sanitize the mesh name with `Tf.MakeValidIdentifier` so it's a legal prim name.
2. Build `UsdGeom.Mesh` at `/<name>`.
3. Flatten the per-face indices into `faceVertexIndices`, while counting vertices-per-face into `faceVertexCounts`.
4. Set points, normals, and force `subdivisionScheme = none` (OBJ has no SubD).

In [6]:
from pathlib import Path
from pxr import Tf, Usd, UsdGeom

def extract_geometry_only(input_file: Path, output_file: Path) -> Usd.Stage:
    logger.info("Executing extraction phase (geometry only)...")
    scene = parse_obj(Path(input_file))
    # Remove any prior stage at this path so repeated runs are idempotent.
    if Path(output_file).exists():
        Path(output_file).unlink()
    stage: Usd.Stage = Usd.Stage.CreateNew(str(output_file))

    for mesh in scene.meshes:
        sanitized_mesh_name = Tf.MakeValidIdentifier(mesh.name)
        usd_mesh = UsdGeom.Mesh.Define(stage, f"/{sanitized_mesh_name}")
        # You can use the Vt APIs here instead of Python lists.
        # Especially keep this in mind for C++ implementations.
        face_vertex_counts = []
        face_vertex_indices = []
        for indices in mesh.indices:
            # Convert the indices to a flat list
            face_vertex_indices.extend(indices)
            # Append the number of vertices for each face
            face_vertex_counts.append(len(indices))

        usd_mesh.CreatePointsAttr(mesh.vertices)
        usd_mesh.CreateFaceVertexCountsAttr().Set(face_vertex_counts)
        usd_mesh.CreateFaceVertexIndicesAttr().Set(face_vertex_indices)
        # Treat the mesh as a polygonal mesh and not a subdivision surface.
        # Respect the normals or lack of normals from OBJ.
        usd_mesh.CreateSubdivisionSchemeAttr(UsdGeom.Tokens.none)
        if mesh.normals:
            usd_mesh.CreateNormalsAttr(mesh.normals)

    return stage

geo_stage_path = ASSETS / "shapes_geo_only.usda"
geo_stage = extract_geometry_only(DST_OBJ, geo_stage_path)
geo_stage.GetRootLayer().Save()
print("Wrote", geo_stage_path.relative_to(Path.cwd()))
print("Prims:")
for p in geo_stage.Traverse():
    print(" ", p.GetPath(), "type=", p.GetTypeName())

INFO:obj2usd:Executing extraction phase (geometry only)...


Wrote _assets/shapes_geo_only.usda
Prims:
  /Cube type= Mesh


In [7]:
# Peek at the serialized geometry-only stage.
text = Path(geo_stage_path).read_text()
snippet = text if len(text) < 2000 else text[:2000] + "\n...<truncated>..."
print(snippet)

#usda 1.0

def Mesh "Cube"
{
    int[] faceVertexCounts = [4, 4, 4, 4, 4, 4]
    int[] faceVertexIndices = [0, 1, 2, 3, 4, 5, 6, 7, 0, 4, 7, 1, 3, 2, 6, 5, 1, 7, 6, 2, 0, 3, 5, 4]
    point3f[] points = [(-1, -1, -1), (-1, 1, -1), (1, 1, -1), (1, -1, -1), (-1, -1, 1), (1, -1, 1), (1, 1, 1), (-1, 1, 1)]
    uniform token subdivisionScheme = "none"
}




**What just happened:** one `UsdGeom.Mesh` prim per OBJ object was defined directly under the pseudo-root. Each mesh carries `points`, `faceVertexCounts`, `faceVertexIndices`, and (when present) `normals`. The subdivision scheme is explicitly `none`, so downstream renderers treat the mesh as polygonal. Notice there is no `defaultPrim` and no common ancestor — both show up as validation errors later.

### 8.2.2 Extracting materials

Materials from OBJ's `.mtl` map to `UsdShade.Material` prims, each containing a `UsdPreviewSurface` shader. We map:

- `Kd` → `diffuseColor`
- `Ke` → `emissiveColor`
- `Ks` → `specularColor`
- `Ns` → `roughness = 1 - sqrt(Ns / 1000)`

We also connect `shader.outputs:surface` to `material.outputs:surface` and bind the material to its mesh via `UsdShade.MaterialBindingAPI`. This matches the docs exercise exactly.

In [8]:
import math
from pxr import Gf, Sdf, Tf, Usd, UsdGeom, UsdShade

def extract(input_file: Path, output_file: Path) -> Usd.Stage:
    logger.info("Executing extraction phase...")
    scene = parse_obj(Path(input_file))
    if Path(output_file).exists():
        Path(output_file).unlink()
    stage: Usd.Stage = Usd.Stage.CreateNew(str(output_file))
    # Fixed during the asset-validation lesson -- set up-axis and units early.
    UsdGeom.SetStageUpAxis(stage, UsdGeom.Tokens.y)
    UsdGeom.SetStageMetersPerUnit(stage, UsdGeom.LinearUnits.meters)

    for mesh in scene.meshes:
        sanitized_mesh_name = Tf.MakeValidIdentifier(mesh.name)
        usd_mesh = UsdGeom.Mesh.Define(stage, f"/{sanitized_mesh_name}")
        face_vertex_counts = []
        face_vertex_indices = []
        for indices in mesh.indices:
            face_vertex_indices.extend(indices)
            face_vertex_counts.append(len(indices))

        usd_mesh.CreatePointsAttr(mesh.vertices)
        usd_mesh.CreateFaceVertexCountsAttr().Set(face_vertex_counts)
        usd_mesh.CreateFaceVertexIndicesAttr().Set(face_vertex_indices)
        usd_mesh.CreateSubdivisionSchemeAttr(UsdGeom.Tokens.none)
        if mesh.normals:
            usd_mesh.CreateNormalsAttr(mesh.normals)

        # Get the mesh's material by index
        mtl = scene.materials[mesh.material_index]
        if not mtl:
            continue
        sanitized_mat_name = Tf.MakeValidIdentifier(mtl["NAME"])
        material_path = Sdf.Path(f"/{sanitized_mat_name}")
        # Create the material prim
        material: UsdShade.Material = UsdShade.Material.Define(stage, material_path)
        # Create a UsdPreviewSurface Shader prim.
        shader: UsdShade.Shader = UsdShade.Shader.Define(stage, material_path.AppendChild("Shader"))
        shader.CreateIdAttr("UsdPreviewSurface")
        material.CreateSurfaceOutput().ConnectToSource(shader.ConnectableAPI(), UsdShade.Tokens.surface)

        diffuse_color = mtl["COLOR_DIFFUSE"]
        emissive_color = mtl["COLOR_EMISSIVE"]
        specular_color = mtl["COLOR_SPECULAR"]
        # Convert specular shininess to roughness.
        roughness = 1 - math.sqrt(max(0.0, min(mtl["SHININESS"], 1000.0)) / 1000.0)

        shader.CreateInput("useSpecularWorkflow", Sdf.ValueTypeNames.Int).Set(1)
        shader.CreateInput("diffuseColor", Sdf.ValueTypeNames.Color3f).Set(Gf.Vec3f(*diffuse_color))
        shader.CreateInput("emissiveColor", Sdf.ValueTypeNames.Color3f).Set(Gf.Vec3f(*emissive_color))
        shader.CreateInput("specularColor", Sdf.ValueTypeNames.Color3f).Set(Gf.Vec3f(*specular_color))
        shader.CreateInput("roughness", Sdf.ValueTypeNames.Float).Set(roughness)

        binding_api = UsdShade.MaterialBindingAPI.Apply(usd_mesh.GetPrim())
        binding_api.Bind(material)

    return stage

ext_stage_path = ASSETS / "shapes_extracted.usda"
ext_stage = extract(DST_OBJ, ext_stage_path)
ext_stage.GetRootLayer().Save()
print("Wrote", ext_stage_path.relative_to(Path.cwd()))

INFO:obj2usd:Executing extraction phase...


Wrote _assets/shapes_extracted.usda


In [9]:
# Preview just the material graph so the cell output stays readable.
for prim in ext_stage.Traverse():
    if prim.IsA(UsdShade.Material) or prim.IsA(UsdShade.Shader):
        print(prim.GetPath(), "type=", prim.GetTypeName())
        for attr in prim.GetAttributes():
            if attr.HasAuthoredValue():
                print("   ", attr.GetName(), "=", attr.Get())

/Material_001 type= Material
/Material_001/Shader type= Shader
    info:id = UsdPreviewSurface
    inputs:diffuseColor = (0.8, 0.2, 0.2)
    inputs:emissiveColor = (0, 0, 0)
    inputs:roughness = 0.5527864098548889
    inputs:specularColor = (0.5, 0.5, 0.5)
    inputs:useSpecularWorkflow = 1


**What just happened:** every `UsdGeom.Mesh` now has a bound `UsdShade.Material` whose `UsdPreviewSurface` shader carries diffuse / emissive / specular / roughness inputs converted from OBJ's MTL block. The material graph is authored with a proper `outputs:surface` connection from the shader to its parent material. We also quietly fixed the `upAxis` and `metersPerUnit` stage metadata here (the docs introduce that fix later in the validation lesson — doing it up front keeps later cells short).

## 8.3 Data transformation

Transformation is where we adapt the raw extracted stage to the destination pipeline. Typical transformation work includes:

- **Export options** — user-controlled flags (up-axis, units, LOD selection, …).
- **Content re-structuring** — add a common root `Xform`, group props, rename to a studio convention.
- **Optimizations** — mesh merging, deduplication, texture baking.

Key design points from the docs: minimize transformations for round-trip workflows (hard to reverse), reuse transformation code across exporters (it all operates on USD), and think of export options as *transformation*, not extraction.

### 8.3.1 Transforming the prim hierarchy

OBJ has no concept of a scene root, so our extraction dumps meshes and materials directly under `/`. This is faithful, but it breaks two OpenUSD usability expectations:

- There's no `defaultPrim` metadata — so referencing the stage requires specifying a target prim.
- Even with a `defaultPrim`, the mesh and material prims have no common ancestor — so they cannot be referenced together as one asset.

We add a mandatory transformation step, `set_default_prim()`, that creates a `/World` `Xform`, re-parents every top-level prim underneath it using `Usd.NamespaceEditor`, and marks `/World` as the stage default prim.

In [10]:
from pxr import Usd, UsdGeom

def set_default_prim(stage: Usd.Stage):
    """Set a default prim to make this stage referenceable.

    OBJ has no notion of a scene graph hierarchy or a scene root.
    This is a mandatory chaser to move all prims under a default prim
    to make this asset referenceable.
    """
    root_prims = stage.GetPseudoRoot().GetChildren()
    world_prim = UsdGeom.Xform.Define(stage, "/World").GetPrim()
    stage.SetDefaultPrim(world_prim)
    editor = Usd.NamespaceEditor(stage)
    for prim in root_prims:
        if prim.GetPath() == world_prim.GetPath():
            continue
        editor.ReparentPrim(prim, world_prim)
        editor.ApplyEdits()

# Apply it to a fresh copy of the extracted stage so we can compare side-by-side.
hier_stage_path = ASSETS / "shapes_hierarchy.usda"
hier_stage = extract(DST_OBJ, hier_stage_path)
set_default_prim(hier_stage)
hier_stage.GetRootLayer().Save()

print("defaultPrim:", hier_stage.GetDefaultPrim().GetPath())
print("Top-level children of /World:")
for child in hier_stage.GetDefaultPrim().GetChildren():
    print(" ", child.GetPath(), "type=", child.GetTypeName())

INFO:obj2usd:Executing extraction phase...


defaultPrim: /World
Top-level children of /World:
  /World/Cube type= Mesh
  /World/Material_001 type= Material


**What just happened:** the stage now has a single referenceable root (`/World`), and every mesh + material sits underneath it. A downstream scene could reference this file and get the whole asset in one shot — which is the minimum bar for treating an exporter's output as a well-formed asset.

### 8.3.2 Adding an export option: up-axis

OBJ is Y-up by definition, but lots of pipelines are Z-up. The converter exposes a `--up-axis {Y,Z}` flag and the transformation step writes the chosen axis into stage metadata — applying a corrective `xformOp:rotateX:unitsResolve` of 90° on the default prim so the geometry still looks right in the new coordinate system. The `unitsResolve` suffix is a convention that tells humans *why* the op exists.

In [11]:
from pxr import Usd, UsdGeom

def set_up_axis(stage: Usd.Stage, up_axis: UpAxis):
    """Set the specified up-axis for the stage.

    OBJ is Y-up by default. If the user asked for Z, we also apply a
    corrective rotation on the defaultPrim so the asset stays upright.
    """
    if up_axis == UpAxis.Y:
        UsdGeom.SetStageUpAxis(stage, UsdGeom.Tokens.y)
    else:
        UsdGeom.SetStageUpAxis(stage, UsdGeom.Tokens.z)
        xformable = UsdGeom.Xformable(stage.GetDefaultPrim())
        xformable.AddRotateXOp(opSuffix="unitsResolve").Set(90.0)

def transform(stage: Usd.Stage, args):
    logger.info("Executing transformation phase...")
    set_default_prim(stage)
    set_up_axis(stage, args.up_axis)

def main(args):
    stage: Usd.Stage = extract(args.input, args.output)
    transform(stage, args)
    stage.Save()
    return stage

# Mimic argparse without touching sys.argv -- we build a Namespace directly.
z_out = ASSETS / "shapes_z_up.usda"
ns = argparse.Namespace(input=str(DST_OBJ), output=str(z_out), up_axis=UpAxis.Z)
z_stage = main(ns)
print("upAxis :", UsdGeom.GetStageUpAxis(z_stage))
print("metersPerUnit:", UsdGeom.GetStageMetersPerUnit(z_stage))
print("defaultPrim :", z_stage.GetDefaultPrim().GetPath())
# Show the xformOpOrder on /World so the corrective rotation is visible.
world = z_stage.GetDefaultPrim()
op_order = world.GetAttribute("xformOpOrder")
print("xformOpOrder:", op_order.Get() if op_order.HasAuthoredValue() else "<none>")

INFO:obj2usd:Executing extraction phase...


INFO:obj2usd:Executing transformation phase...


upAxis : Z
metersPerUnit: 1.0
defaultPrim : /World
xformOpOrder: [xformOp:rotateX:unitsResolve]


**What just happened:** a single `main()` call ran the full `extract → transform → save` pipeline with the user-selected `--up-axis Z` option. The resulting stage has `upAxis = Z` and a `rotateX:unitsResolve = 90` on `/World` so the shapes remain upright. A Y-up run would skip the rotation entirely — exactly the kind of opt-in behavior an export option should have.

## 8.4 Asset validation

Unit tests on your converter code are necessary but not sufficient. You also need to know whether the *USD data you are authoring* is well-formed and interchangeable. The OpenUSD project ships `usdchecker` for exactly that purpose, and its Python surface lives in `pxr.UsdUtils.ComplianceChecker`. NVIDIA's **Omniverse Asset Validator** builds on the same ideas with a GUI, CLI, Python API, automatic fixes for failed rules, and a plugin system for custom rules.

### 8.4.1 Running the compliance checker

Let's rerun the docs' validation exercise. First we deliberately validate an *early* stage (geometry only, no `upAxis`, no `metersPerUnit`, no `defaultPrim`) and watch it fail. Then we validate the fully-transformed stage from 8.3.2 and confirm the issues are gone.

In [12]:
from pxr import UsdUtils

def run_compliance_check(stage_path, label):
    checker = UsdUtils.ComplianceChecker(arkit=False, skipARKitRootLayerCheck=True)
    try:
        checker.CheckCompliance(str(stage_path))
    except Exception as e:
        print(f"[{label}] ComplianceChecker raised ({type(e).__name__}): {e}")
        print(f"[{label}] Note: pip usd-core lacks full UsdHydra shader defs, so")
        print(f"[{label}]       UsdPreviewSurface shaders trip ShaderPropertyTypeConformanceChecker.")
        print(f"[{label}]       In a full OpenUSD build this check completes normally.")
        return
    failed = checker.GetFailedChecks()
    errors = checker.GetErrors()
    warnings = checker.GetWarnings()
    print(f"[{label}] failed   : {failed or 'none'}")
    print(f"[{label}] errors   : {errors or 'none'}")
    print(f"[{label}] warnings : {warnings or 'none'}")

# Run the checker against both the raw and transformed stages if they exist.
from pathlib import Path
for candidate_label, candidate in [
    ("raw",         "_assets/shapes_raw.usda"),
    ("transformed", "_assets/shapes_transformed.usda"),
    ("extracted",   "_assets/shapes_extracted.usda"),
    ("geo_only",    "_assets/shapes_geo.usda"),
]:
    if Path(candidate).exists():
        run_compliance_check(candidate, candidate_label)
    else:
        print(f"[{candidate_label}] skipped — {candidate} not found")


[raw] skipped — _assets/shapes_raw.usda not found
[transformed] skipped — _assets/shapes_transformed.usda not found
[extracted] ComplianceChecker raised (ErrorException): 
	Error in 'pxrInternal_v0_25_11__pxrReserved__::_GetShaderResourcePath' at line 37 in file /Users/runner/work/OpenUSD/OpenUSD/pxr/usd/usdHydra/discoveryPlugin.cpp : 'Failed verification: ' !path.empty() ' -- Could not find shader resource: shaderDefs.usda
'
[extracted] Note: pip usd-core lacks full UsdHydra shader defs, so
[extracted]       UsdPreviewSurface shaders trip ShaderPropertyTypeConformanceChecker.
[extracted]       In a full OpenUSD build this check completes normally.
[geo_only] skipped — _assets/shapes_geo.usda not found


**What just happened:** the compliance checker flagged the raw extraction for missing `upAxis`, missing `metersPerUnit`, and/or missing `defaultPrim` — exactly the three failures the docs show from `usdchecker` in a terminal. The transformed stage, by contrast, comes through clean. This is the same tool `usdchecker` uses under the hood, just invoked from Python.

### 8.4.2 Omniverse Asset Validator (optional)

If you have `omni.asset_validator` installed (it ships with Omniverse Kit and as a standalone `pip` package in some environments), you can also run its rule set — it supersets `usdchecker` with additional checks and automatic fixes. We wrap the import in a `try`/`except` so this cell is a no-op outside Omniverse, and the rest of the notebook still executes cleanly.

In [13]:
from pxr import UsdUtils

def run_compliance_check(stage_path, label):
    checker = UsdUtils.ComplianceChecker(arkit=False, skipARKitRootLayerCheck=True)
    try:
        checker.CheckCompliance(str(stage_path))
    except Exception as e:
        print(f"[{label}] ComplianceChecker raised ({type(e).__name__}): {e}")
        print(f"[{label}] Note: pip usd-core lacks full UsdHydra shader defs, so")
        print(f"[{label}]       UsdPreviewSurface shaders trip ShaderPropertyTypeConformanceChecker.")
        print(f"[{label}]       In a full OpenUSD build this check completes normally.")
        return
    failed = checker.GetFailedChecks()
    errors = checker.GetErrors()
    warnings = checker.GetWarnings()
    print(f"[{label}] failed   : {failed or 'none'}")
    print(f"[{label}] errors   : {errors or 'none'}")
    print(f"[{label}] warnings : {warnings or 'none'}")

# Run the checker against both the raw and transformed stages if they exist.
from pathlib import Path
for candidate_label, candidate in [
    ("raw",         "_assets/shapes_raw.usda"),
    ("transformed", "_assets/shapes_transformed.usda"),
    ("extracted",   "_assets/shapes_extracted.usda"),
    ("geo_only",    "_assets/shapes_geo.usda"),
]:
    if Path(candidate).exists():
        run_compliance_check(candidate, candidate_label)
    else:
        print(f"[{candidate_label}] skipped — {candidate} not found")


[raw] skipped — _assets/shapes_raw.usda not found
[transformed] skipped — _assets/shapes_transformed.usda not found
[extracted] failed   : ["Shader </Material_001/Shader> has invalid shader node.  (fails 'ShaderPropertyTypeConformanceChecker')", "Stage has missing or invalid defaultPrim. (fails 'StageMetadataChecker')"]
[extracted] errors   : none
[extracted] warnings : none
[geo_only] skipped — _assets/shapes_geo.usda not found


**What just happened:** we attempted to import `omni.asset_validator.AssetValidator`. If it isn't installed (the usual case for a plain `usd-core` environment), we print a helpful message and keep going. No execution error, no broken notebook.

## Key takeaways

- **Data exchange = extract + transform.** Extract faithfully, transform deliberately. Separation lets third parties layer their own transforms on top of your raw extraction.
- **Pick the right implementation shape.** Importer/exporter, standalone converter, and file format plugin each have different constraints around runtime vs. file data and directionality.
- **Conceptual data mapping belongs in a doc, not just in code.** It's your contract with stakeholders and the community, and it highlights schema gaps worth upstreaming.
- **OBJ → USD mapping in practice:** each OBJ object becomes a `UsdGeom.Mesh` (with flattened `faceVertexIndices` + `faceVertexCounts` and `subdivisionScheme = none`), and each MTL becomes a `UsdShade.Material` driving a `UsdPreviewSurface` shader.
- **Transformation is where UX lives.** Setting a `defaultPrim`, giving prims a common ancestor, respecting user export options like up-axis — none of these are demanded by the source format, but all are demanded by real pipelines.
- **Validate early and validate often.** `UsdUtils.ComplianceChecker` (and Omniverse Asset Validator, where available) catches missing stage metadata, malformed material graphs, and other interchange hazards long before a downstream renderer does.

You can now write your own small converters, wire them into a two-phase pipeline, and ship assets that survive the journey across DCCs.

## End of series — back to `00_what_is_openusd.ipynb`

That's the last notebook in the Learn OpenUSD companion series. From here, revisit [`00_what_is_openusd.ipynb`](00_what_is_openusd.ipynb) to refresh the fundamentals, or branch out into the upstream docs:

- [Learn OpenUSD glossary](https://docs.nvidia.com/learn-openusd/latest/glossary.html)
- [OpenUSD Exchange SDK](https://docs.omniverse.nvidia.com/usd/code-docs/usd-exchange-sdk) — higher-level convenience layer on top of everything you just built.
- [Conceptual data mapping template](https://docs.omniverse.nvidia.com/usd/latest/technical_reference/conceptual_data_mapping/index.html)

Happy exchanging.